# 14 — Convolution: one small filter, slid over an image

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/014-architecture-families.ipynb)

A convolution layer owns a handful of tiny filters and slides each one over the image, multiplying and adding at every position. This notebook applies a 3×3 edge detector by hand, draws the resulting response map, lets you move the filter with a slider, and ends by shuffling the pixels to show why neighbourhoods matter.

## 1. The numbers: an 8×8 image and a 3×3 filter

The image is a dark left half and a bright right half, so there is one vertical edge between columns 3 and 4. The filter subtracts the left column of its window from the right column, which means it responds strongly exactly where brightness rises.

In [ ]:
import numpy as np                                     # arrays for the image

image = np.full((8, 8), 0.15)                          # an 8x8 grey image, dark background
image[:, 4:] = 0.85                                    # the right half is bright: an edge at column 4
image[2, 1] = 0.90                                     # one stray bright pixel, as real photos have
print(image.round(2))                                  # eight rows of eight numbers

k = np.array([[-1., 0., 1.],                           # filter row 1: dark on the left, bright on the right
              [-1., 0., 1.],                           # filter row 2: the same
              [-1., 0., 1.]])                          # filter row 3: the same
patch = image[3:6, 3:6]                                # the 3x3 window sitting on the edge
print("patch:")                                        # a label for the next print
print(patch)                                           # what the filter sees at that spot
print("response:", np.sum(patch * k))                  # 2.1: three brights minus three darks

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — NumPy, preinstalled on Colab.</li><li><code>image = np.full((8, 8), 0.15)</code> — an 8×8 array filled with the value 0.15.</li><li><code>image[:, 4:] = 0.85</code> — every row, columns 4 onward, set to bright.</li><li><code>image[2, 1] = 0.90</code> — a single bright pixel at row 2, column 1.</li><li><code>print(image.round(2))</code> — rounds for readability, then prints the grid.</li><li><code>k = np.array([...])</code> — the 3×3 filter: −1 down the left, +1 down the right.</li><li><code>patch = image[3:6, 3:6]</code> — rows 3–5 and columns 3–5, a 3×3 crop.</li><li><code>np.sum(patch * k)</code> — multiply matching cells, then add all nine products.</li></ul>

</details>

## 2. Step by step: slide the filter over every position

A 3×3 filter placed on an 8×8 image has 6×6 legal positions, so the output is a 6×6 map. Two nested loops visit one position per iteration; the printed row appears as soon as that row is complete.

In [ ]:
size = image.shape[0] - k.shape[0] + 1                 # 8 - 3 + 1 = 6 output positions per side
response = np.zeros((size, size))                      # the 6x6 response map, filled below

for i in range(size):                                  # one output row at a time
    for j in range(size):                              # one output column at a time
        window = image[i:i + 3, j:j + 3]               # the 3x3 patch under this position
        response[i, j] = np.sum(window * k)            # nine products, added into one number
    print(f"row {i}:", response[i].round(2))           # print the row the moment it is finished

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>size = ...</code> — output side length; a 3×3 filter removes one row and one column from each side.</li><li><code>response = np.zeros(...)</code> — an empty 6×6 map to fill in.</li><li><code>for i in range(size):</code> — the output row index.</li><li><code>for j in range(size):</code> — the output column index.</li><li><code>window = image[i:i + 3, j:j + 3]</code> — the 3×3 neighbourhood at this position.</li><li><code>response[i, j] = np.sum(window * k)</code> — elementwise product, then sum: the response.</li><li><code>print(...)</code> — one line per finished row, so the map builds up in the output.</li></ul>

</details>

## 3. Plot: the image and its response side by side

The left panel is the input, the right panel is what the filter reports at each position. The response is high only along the edge and zero on the flat halves: the filter is blind to uniform brightness, which is what makes it an edge detector.

In [ ]:
import matplotlib.pyplot as plt                        # the plotting library

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))       # one row, two panels
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)     # left: the input image
axes[0].set_title("input: one vertical edge")          # panel title
axes[0].set_xlabel("column")                           # x axis label
axes[0].set_ylabel("row")                              # y axis label

pic = axes[1].imshow(response, cmap="RdBu_r", vmin=-2.5, vmax=2.5)   # right: the response, red = positive
axes[1].set_title("response: bright along the edge")   # panel title
axes[1].set_xlabel("filter column")                    # x axis label
axes[1].set_ylabel("filter row")                       # y axis label
fig.colorbar(pic, ax=axes[1], label="sum of patch x filter")   # the scale for the colours
fig.suptitle("the same nine weights fire everywhere the edge passes")   # the takeaway
plt.tight_layout()                                     # stop the labels colliding
plt.show()                                             # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — Colab ships matplotlib.</li><li><code>fig, axes = plt.subplots(1, 2, ...)</code> — a figure with a left and a right panel.</li><li><code>axes[0].imshow(image, cmap="gray")</code> — draw the image in greyscale.</li><li><code>axes[0].set_title(...)</code> / <code>set_xlabel(...)</code> / <code>set_ylabel(...)</code> — label the input panel.</li><li><code>axes[1].imshow(response, cmap="RdBu_r")</code> — a diverging colour map: red positive, blue negative.</li><li><code>vmin=-2.5, vmax=2.5</code> — fix the colour scale so both signs are visible.</li><li><code>axes[1].set_...</code> — label the response panel and both of its axes.</li><li><code>fig.colorbar(...)</code> — a scale showing what each colour means.</li><li><code>fig.suptitle(...)</code> — the one-sentence takeaway.</li><li><code>plt.tight_layout()</code> / <code>plt.show()</code> — tidy the spacing, then draw.</li></ul>

</details>

## 4. Interactive: put the filter where you like

The slider walks the filter through all 36 positions in reading order, and the red box shows exactly which nine pixels it is reading. The title prints the response at that position, so you can find the edge without moving the mouse.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(pos=20):                                      # pos runs over the 36 output positions
    i, j = divmod(pos, 6)                              # row-major index -> row i, column j
    fig, ax = plt.subplots(figsize=(4.2, 3.6))         # a fresh square-ish figure per drag
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)      # the image underneath
    box = plt.Rectangle((j - 0.5, i - 0.5), 3, 3, fill=False,   # corner of the box, and its size
                        edgecolor="tab:red", linewidth=2)      # hollow, with a red outline
    ax.add_patch(box)                                  # draw the window on top
    ax.set_xlabel("column")                            # x axis label
    ax.set_ylabel("row")                               # y axis label
    ax.set_title(f"position ({i},{j})   response = {response[i, j]:.2f}")   # live readout
    plt.show()                                         # draw the frame

interact(show, pos=(0, 35, 1))                         # slider over the 36 positions

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the widget helper.</li><li><code>def show(pos=20)</code> — redraws one frame at the chosen position.</li><li><code>i, j = divmod(pos, 6)</code> — turns a single number into a row and a column.</li><li><code>plt.subplots(...)</code> — a new figure each time, so the box does not pile up.</li><li><code>ax.imshow(...)</code> — draws the 8×8 image.</li><li><code>plt.Rectangle(...)</code> — a hollow 3×3 box, with no fill so the pixels stay visible.</li><li><code>ax.add_patch(box)</code> — adds the box to the axes.</li><li><code>ax.set_...</code> — axis labels and the title showing the response.</li><li><code>interact(show, pos=(0, 35, 1))</code> — the slider from position 0 to 35.</li></ul>

</details>

## 5. When it breaks: shuffle the pixels

Shuffling the 64 brightness values keeps the image's histogram exactly the same and destroys only their arrangement. The filter's peak response and its total energy both collapse, because subtraction of neighbours now hits random pairs instead of the edge. This is why a convolution cannot be replaced by a per-pixel lookup table.

In [ ]:
rng = np.random.default_rng(0)                         # a fixed seed, so the shuffle repeats
shuffled = image.flatten().copy()                      # the same 64 values, in one flat row
rng.shuffle(shuffled)                                  # random order: same values, no structure
shuffled = shuffled.reshape(8, 8)                      # back to an 8x8 image

response_shuffled = np.zeros((6, 6))                   # a fresh response map
for i in range(6):                                     # output row
    for j in range(6):                                 # output column
        response_shuffled[i, j] = np.sum(shuffled[i:i + 3, j:j + 3] * k)   # same filter, shuffled input

print("max |response|, edge     :", np.abs(response).max().round(2))          # 2.10
print("max |response|, shuffled :", np.abs(response_shuffled).max().round(2)) # 1.45
print("response energy, edge    :", np.sum(response ** 2).round(2))           # 54.61
print("response energy, shuffled:", np.sum(response_shuffled ** 2).round(2))  # 15.76

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))       # two panels
axes[0].imshow(shuffled, cmap="gray", vmin=0, vmax=1)  # left: the shuffled image
axes[0].set_xlabel("column")                           # x axis label
axes[0].set_ylabel("row")                              # y axis label
axes[0].set_title("shuffled pixels: no edge left")     # panel title
axes[1].imshow(response_shuffled, cmap="RdBu_r", vmin=-2.5, vmax=2.5)   # right: its response
axes[1].set_xlabel("filter column")                    # x axis label
axes[1].set_ylabel("filter row")                       # y axis label
axes[1].set_title("response: no coherent line")        # panel title
fig.suptitle("convolution reads neighbourhoods, not pixel values")      # the takeaway
plt.tight_layout()                                     # tidy the spacing
plt.show()                                             # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>rng = np.random.default_rng(0)</code> — a seeded generator so the failure is reproducible.</li><li><code>shuffled = image.flatten().copy()</code> — one flat list of all 64 brightness values.</li><li><code>rng.shuffle(shuffled)</code> — reorders them in place.</li><li><code>shuffled.reshape(8, 8)</code> — folds the list back into an image.</li><li><code>response_shuffled = np.zeros((6, 6))</code> — a new map for the shuffled input.</li><li><code>for i ... for j ...</code> — the same double loop as section 2.</li><li><code>np.sum(shuffled[...] * k)</code> — the identical filter, applied to the scrambled pixels.</li><li><code>print(...)</code> — peak response and total energy for both images.</li><li><code>axes[0].imshow(shuffled, ...)</code> — the scrambled image, which looks like noise.</li><li><code>axes[1].imshow(response_shuffled, ...)</code> — its response, with no line along the edge.</li><li><code>fig.suptitle(...)</code> — the takeaway: the filter needs the arrangement, not the values.</li></ul>

</details>

## Your turn

In section 4, set the slider to a position on the flat dark half and then on the edge: what response do you predict for each? Then replace `k` with a horizontal detector (`[[-1,-1,-1],[0,0,0],[1,1,1]]`) and predict, before rerunning, what the right-hand panel of section 3 will show.